# 250 · A continuous wave inside an experiment

**Level:** medium · **Topics:** Gen2, SHFQC, dsl, pulses, sweeps, spectroscopy, calibration · **Type:** write the code

A user wants a continuous wave (CW) on the drive line, sweeps its frequency in steps, and measures at every step.
It has to be a LabOne Q `Experiment`, because the whole framework of the user is built on experiments.
A step can be long: the full sweep may last a minute.

The interface is fixed:

```python
def drive_cw(frequency_sweep, length_of_step, integration_length):
    ...
    return experiment
```

An experiment only plays pulses, so there is no true CW. Write the closest thing: at every frequency,
one pulse that lasts `length_of_step`, and a measurement of `integration_length` at the **end** of the step.

Cells with a `TODO` are yours to complete. Everything runs in emulation, no instrument needed.

In [ ]:
from laboneq.contrib.example_helpers.plotting.plot_helpers import plot_simulation
from laboneq.simple import *

## Device setup

One SHFQC+ with one qubit: a drive line and a readout line.

📖 [Device setup](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/00_device_setup/concepts/index.html)

In [ ]:
setup = DeviceSetup("exercise_setup")
setup.add_dataserver(host="localhost", port="8004")
setup.add_instruments(
    SHFQC(uid="shfqc", address="DEV12001", device_options="SHFQC/PLUS/QC2CH")
)
setup.add_connections(
    "shfqc",
    create_connection(to_signal="q0/drive", ports="SGCHANNELS/0/OUTPUT"),
    create_connection(to_signal="q0/measure", ports="QACHANNELS/0/OUTPUT"),
    create_connection(to_signal="q0/acquire", ports="QACHANNELS/0/INPUT"),
)

## Calibration

📖 [Calibrating signals](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/02_logical_signals/concepts/00_calibrating_signals.html)

In [ ]:
calibration = Calibration()
calibration["q0/drive"] = SignalCalibration(
    oscillator=Oscillator(frequency=100e6, modulation_type=ModulationType.HARDWARE),
    local_oscillator=Oscillator(frequency=5.0e9),
    range=0,
)
calibration["q0/measure"] = SignalCalibration(
    oscillator=Oscillator(frequency=50e6),
    local_oscillator=Oscillator(frequency=7.0e9),
    range=0,
)
calibration["q0/acquire"] = SignalCalibration(
    oscillator=Oscillator(frequency=50e6),
    local_oscillator=Oscillator(frequency=7.0e9),
    range=0,
)
setup.set_calibration(calibration)

## Session

📖 [Session](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/01_session/concepts/index.html)

In [ ]:
session = Session(setup)
session.connect(do_emulation=True)

## The experiment

Three things are needed:

1. a constant pulse of `length_of_step` that fits in the memory of the instrument, whatever its length,
2. a sweep of the **frequency of the drive oscillator**: the oscillator is part of the calibration, so the sweep parameter goes into the calibration of the experiment,
3. the measurement at the end of each step.

📖 [Pulse commands](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/03_sections_pulses/concepts/00_pulse_commands.html) ·
[Frequency sweep examples](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/03_sections_pulses/tutorials/03_frequency_sweep_examples.html) ·
[Experiment calibration](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/05_experiment/concepts/00_experiment_calibration.html) ·
[Timing rules](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/03_sections_pulses/concepts/01_timing_rules.html)

In [ ]:
def drive_cw(frequency_sweep, length_of_step, integration_length):
    # TODO: a constant pulse that lasts `length_of_step`
    cw_pulse = None
    readout_pulse = pulse_library.const(length=integration_length, amplitude=0.5)

    @dsl.experiment(signals=["drive", "measure", "acquire"])
    def cw_sweep():
        with dsl.acquire_loop_rt(count=1):
            # TODO: sweep `frequency_sweep`
            # TODO: play `cw_pulse` on "drive"
            # TODO: place the measurement at the end of the step
            with dsl.section(name="step"):
                dsl.measure(
                    measure_signal="measure",
                    acquire_signal="acquire",
                    measure_pulse=readout_pulse,
                    integration_kernel=readout_pulse,
                    handle="result",
                )

        # TODO: make the oscillator of "drive" follow the sweep
        experiment_calibration = dsl.experiment_calibration()

    experiment = cw_sweep()
    experiment.set_signal_map(
        {
            "drive": "q0/drive",
            "measure": "q0/measure",
            "acquire": "q0/acquire",
        }
    )
    return experiment

## Run it

101 frequencies, 1 ms per step, 2 µs of integration.

In [ ]:
frequencies = LinearSweepParameter(start=-200e6, stop=200e6, count=101)

experiment = drive_cw(frequencies, length_of_step=1e-3, integration_length=2e-6)
compiled = session.compile(experiment)
results = session.run(compiled)

In [ ]:
# check
print(results["result"].data.shape)
print(round(compiled.estimated_runtime, 2), "s")

**Expected result:**

```
(101,)
0.1 s
```

## Look at it

With short steps of 10 µs the simulation shows the first two steps: the drive is on during the whole step,
and the readout sits at the end of each step.

In [ ]:
short = session.compile(drive_cw(frequencies, length_of_step=10e-6, integration_length=1e-6))
plot_simulation(short, start_time=0, length=22e-6, signals=["drive", "measure"])

## Questions

1. Is the drive really continuous? Look at the plot between two steps.
2. What is the longest `integration_length` this setup accepts?
3. The user wants a sweep that lasts one minute. Does the experiment still compile? What changes?
4. What would you tell the user?

<details>
<summary>Hint 1</summary>

A long constant pulse does not have to be stored sample by sample: look at the arguments of `pulse_library.const`.
</details>

<details>
<summary>Hint 2</summary>

```python
experiment_calibration["drive"] = SignalCalibration(
    oscillator=Oscillator(frequency=..., modulation_type=ModulationType.HARDWARE)
)
```
</details>

<details>
<summary>Hint 3</summary>

Pulses in the same section start together by default. `dsl.section` has an `alignment` argument.
</details>

The solution is in [solution.ipynb](solution.ipynb).